# Colab T4 GPU Bridge Server (Hardened)
Authenticated FastMCP & REST bridge with localhost binding and Cloudflare Tunnel.

In [ ]:
# Cell 1: Hardware Check
import subprocess, sys
print("Checking GPU availability...")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise SystemError("FATAL: No NVIDIA GPU detected! Enable T4 GPU runtime via: Runtime -> Change runtime type -> T4 GPU.")
print(f"SUCCESS: Detected GPU -> {gpu_check.stdout.strip()}")

In [ ]:
# Cell 2: Dependencies & Cloudflared Setup
!pip install -q "fastmcp>=0.4.0" fastapi uvicorn pydantic torch
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true
print("SUCCESS: Installed FastMCP, FastAPI, Uvicorn, and Cloudflared.")

In [ ]:
# Cell 3: Remote Worker Service (Authenticated FastMCP + REST, No Shell=True)
import os, secrets

bridge_token = os.environ.get("COLAB_BRIDGE_TOKEN") or secrets.token_urlsafe(32)
os.environ["COLAB_BRIDGE_TOKEN"] = bridge_token
with open(".colab_bridge_token", "w", encoding="utf-8") as tf:
    tf.write(bridge_token)
os.chmod(".colab_bridge_token", 0o600)

worker_code = '''import hmac, os, shlex, subprocess, sys, torch
from fastmcp import FastMCP
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel

TOKEN_FILE = ".colab_bridge_token"
API_TOKEN = os.environ.get("COLAB_BRIDGE_TOKEN", "")
if not API_TOKEN and os.path.exists(TOKEN_FILE):
    with open(TOKEN_FILE, "r", encoding="utf-8") as f:
        API_TOKEN = f.read().strip()

ALLOWED_BINARIES = {"nvidia-smi", "python", "python3"}

def verify_bearer_token(authorization: str = Header(default="")) -> None:
    if not API_TOKEN:
        raise HTTPException(status_code=500, detail="Server authentication token not configured")
    scheme, _, token = authorization.partition(" ")
    if scheme.lower() != "bearer" or not hmac.compare_digest(token, API_TOKEN):
        raise HTTPException(status_code=401, detail="Invalid or missing Bearer token")

mcp = FastMCP("colab-t4-gpu")

@mcp.tool()
def gpu_info() -> dict:
    """Get current GPU specifications and VRAM utilization."""
    if not torch.cuda.is_available():
        return {"status": "error", "message": "CUDA unavailable"}
    dev = 0
    return {
        "gpu": torch.cuda.get_device_name(dev),
        "allocated_mb": round(torch.cuda.memory_allocated(dev) / (1024 ** 2), 2),
        "reserved_mb": round(torch.cuda.memory_reserved(dev) / (1024 ** 2), 2),
        "total_mb": round(torch.cuda.get_device_properties(dev).total_memory / (1024 ** 2), 2)
    }

@mcp.tool()
def run_gpu_command(cmd: str) -> dict:
    """Execute an allowlisted command without shell expansion on the Colab instance."""
    try:
        args = shlex.split(cmd)
    except ValueError as exc:
        return {"returncode": 1, "stdout": "", "stderr": f"Argument parse error: {exc}"}
    if not args:
        return {"returncode": 1, "stdout": "", "stderr": "Empty command"}
    exe_name = os.path.basename(args[0])
    if exe_name not in ALLOWED_BINARIES:
        return {"returncode": 126, "stdout": "", "stderr": f"Executable not in allowlist: {exe_name}"}
    res = subprocess.run(args, shell=False, capture_output=True, text=True, timeout=60)
    return {
        "returncode": res.returncode,
        "stdout": res.stdout,
        "stderr": res.stderr
    }

@mcp.tool()
def run_inference(prompt: str) -> dict:
    """Generic GPU inference endpoint template."""
    return {"prompt": prompt, "status": "success", "device": torch.cuda.get_device_name(0)}

class CmdRequest(BaseModel):
    cmd: str

class InferRequest(BaseModel):
    prompt: str

app = mcp.get_app() if hasattr(mcp, 'get_app') else FastAPI()

@app.get("/health", dependencies=[Depends(verify_bearer_token)])
def health():
    return {"status": "ok", "cuda": torch.cuda.is_available(), "info": gpu_info()}

@app.post("/api/run", dependencies=[Depends(verify_bearer_token)])
def api_run(req: CmdRequest):
    out = run_gpu_command(req.cmd)
    if out.get("returncode") == 126:
        raise HTTPException(status_code=403, detail=out.get("stderr"))
    return out

@app.post("/api/infer", dependencies=[Depends(verify_bearer_token)])
def api_infer(req: InferRequest):
    return run_inference(req.prompt)
'''

with open('remote_worker.py', 'w', encoding='utf-8') as f:
    f.write(worker_code)
print("SUCCESS: Generated hardened remote_worker.py")

In [ ]:
# Cell 4: Launch Localhost Server & Cloudflare Tunnel
import os, time, re, subprocess

!pkill -f 'uvicorn remote_worker' || true
!pkill -f 'cloudflared tunnel' || true

server_proc = subprocess.Popen(["uvicorn", "remote_worker:app", "--host", "127.0.0.1", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

token_val = os.environ.get("COLAB_BRIDGE_TOKEN", "")
if tunnel_url:
    print("=" * 60)
    print("COLAB T4 GPU TUNNEL READY (AUTHENTICATED)!")
    print(f"ENDPOINT URL: {tunnel_url}")
    print(f"SSE MCP URL:  {tunnel_url}/sse")
    print("=" * 60)
    print(f"Copy this command and run locally:\n  python connect_colab.py --url {tunnel_url} --token {token_val}")
else:
    print("ERROR: Cloudflare tunnel generation timed out. Check output.")